# LoRA fine-tuning across three GPUs

A LoRA fine-tune of **Qwen2.5-32B-Instruct** on three GPUs of one node, submitted to SUSE AI Factory as
an AI Job from this notebook. torchrun starts one process per GPU; each holds the whole model in bf16
and trains on its own share of every batch, and DistributedDataParallel (DDP) averages the adapter's
gradients across the three at every step.

The run is the **LoRA Distributed Training Test** profile from the
[`nvidia-180g` profile pack](../profile-packs/nvidia-180g/README.md). It reports four checks (every
process trained on its own GPU, the three adapters stayed identical, the loss on records it never
trained on came down, the adapter was saved) and its throughput and GPU memory, and keeps the
adapter on the run's checkpoint volume.

**Needs:** a cluster with AI Factory and the `nvidia-180g` pack, a node with three free GPUs of about
180 GB (GB200, B200), a project whose GPU quota allows three, and Application Collection credentials in
AI Factory's Settings (the PyTorch image). The first run downloads the model (65 GB) to the run's
scratch volume: allow 10 to 30 minutes for it, then about 15 minutes of training.

## 1. The SDK

In [ ]:
%pip install -q "rancher-ai[pandas] @ https://github.com/dstanley/aif/archive/refs/heads/pr/7-designs.tar.gz#subdirectory=sdk/python" matplotlib

## 2. Settings

`CONTEXT` is the cluster's context in your kubeconfig (`kubectl config get-contexts`), `PROJECT` the
project's namespace. The `ENV` values override the profile's defaults; the script's docstring lists
every setting (in the profile, or [`lora_distributed_test.py`](../profile-packs/nvidia-180g/scripts/lora_distributed_test.py)).

In [ ]:
import json, os, time

CONTEXT = "gb200"                  # kubeconfig context of the cluster
PROJECT = "training"               # the project's namespace
PROFILE = "lora-distributed-test"
GPUS = 3
RUN = f"lora-ddp-{time.strftime('%m%d-%H%M')}"

ENV = {
    "MODEL": "Qwen/Qwen2.5-32B-Instruct",   # Qwen/Qwen2.5-14B-Instruct: half the download
    "STEPS": "200",
    "BATCH": "4",                           # sequences per GPU per step
    "SEQ_LEN": "1024",
    "LORA_R": "32",
}

In [ ]:
from rancher_ai import Client

ai = Client(context=CONTEXT, project=PROJECT)
print(ai.whoami())
ai.profiles.list()

## 3. Can the cluster run it?

The same check as the Catalog's: the profile's needs (GPU memory, GPUs on one node, driver, CPU
architecture) against what the nodes report.

In [ ]:
fits, why = ai.profiles.fits(PROFILE)
print("fits" if fits else "does not fit: " + "; ".join(why))

## 4. Submit the run

`runs.create` checks the request against the profile's limits and the cluster before it submits. The
run then waits in the project's queue until three GPUs are free.

In [ ]:
run = ai.runs.create(profile=PROFILE, name=RUN, gpus_per_worker=GPUS, env=ENV, runtime_hours=4)
print(run.name, "->", run.dashboard())

In [ ]:
state = run.wait(until={"Running", "Completed", "Failed", "Cancelled"}, timeout=3600)
print(state)
run.status()

## 5. Follow it

Rank 0's log: the libraries installing, the model downloading, then a line every ten steps with the
loss and the tokens per second, in all and per GPU. The cell returns when the run ends; stopping the
cell does not stop the run.

In [ ]:
run.logs(follow=True, tail=50)

## 6. The result

In [ ]:
import pandas as pd

state = run.wait(timeout=6 * 3600)
res = run.result() or {}
print(state, "-", res.get("status", "no result"))
display(pd.DataFrame(res.get("checks", []))[["name", "ok", "detail"]] if res else None)
pd.Series(res.get("metrics", {}), name="metric").to_frame()

## 7. Loss and throughput, step by step

The run's `training.json`, from its checkpoint volume: the settings, each GPU's peak memory, and the
training loss and throughput of every tenth step. Training loss is over different records at each
step, so it is noisy; the held-out loss in the result is the measure of learning.

In [ ]:
vol = ai.checkpoints.get(f"{RUN}-checkpoints")
path = vol.get(f"{RUN}/training.json", "./")
record = json.load(open(path))
steps = pd.DataFrame(record["steps"])
display(pd.DataFrame(record["ranks"])[["rank", "gpu", "memory_gib", "peak_gpu_memory_gib"]])
steps.tail()

In [ ]:
import matplotlib.pyplot as plt

fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4))
a.plot(steps.step, steps.loss, marker="o"); a.set(title="Training loss", xlabel="step", ylabel="loss (answer tokens)")
b.plot(steps.step, steps.tokens_per_s, marker="o", label="all GPUs")
b.plot(steps.step, steps.tokens_per_s_per_gpu, marker="o", label="per GPU")
b.set(title="Throughput", xlabel="step", ylabel="tokens/s"); b.legend()
plt.tight_layout()

## 8. Scaling: one GPU against three (optional)

The same run on one GPU shows what DDP buys: with the same `BATCH` per GPU, three GPUs train on three
times the sequences per step in about the same step time, so tokens per second should be close to
three times one GPU's. Fewer steps keep it short.

In [ ]:
one = ai.runs.create(profile=PROFILE, name=f"{RUN}-1gpu", gpus_per_worker=1, env={**ENV, "STEPS": "60"}, runtime_hours=4)
one.wait(timeout=6 * 3600)
r1, r3 = (one.result() or {}).get("metrics", {}), res.get("metrics", {})
if r1 and r3:
    print(f"1 GPU : {r1['tokens_per_s']:.0f} tokens/s")
    print(f"{r3['gpus']} GPUs: {r3['tokens_per_s']:.0f} tokens/s, {r3['tokens_per_s'] / r1['tokens_per_s']:.2f}x")

## 9. The adapter

It is in the run's checkpoint volume, `<run>/adapter/`, kept after the run. Load it with PEFT onto the
same base model to use it, or download it:

```python
vol.ls(f"{RUN}/adapter")
ai.volumes.get(f"{RUN}-checkpoints", f"{RUN}/adapter", "./adapter")   # any size; a folder comes as a tar
```

**Data** (the cluster's AI Jobs → Data) lists the volume; delete it there, or with
`ai.checkpoints.delete(f"{RUN}-checkpoints", confirm=True)`, when it is no longer needed.